# SBA 7(a) Loan Risk — Independent Model Validation

This notebook independently evaluates the selected Logistic
Regression and Gradient Boosting models on the FY2018
approval cohort.

## Chronological evaluation design

- FY2010–2015: Model training
- FY2016: Hyperparameter tuning
- FY2017: Probability calibration
- FY2018: Independent validation
- FY2019: Final holdout testing

## Models

Logistic Regression:
- 13 provisional approval-time predictors
- L2 regularization, C = 1.0
- Sigmoid probability calibration

Gradient Boosting:
- 13 provisional approval-time predictors
- Learning rate: 0.05
- Maximum iterations: 200
- Maximum leaf nodes: 31
- Sigmoid probability calibration

## Evaluation objectives

1. Compare Average Precision and ROC-AUC.
2. Measure Recall and Precision at Top 10% review capacity.
3. Evaluate Brier scores and probability calibration.
4. Assess calibration deciles and temporal stability.
5. Examine uncertainty in paired model differences.
6. Apply the predefined challenger-selection criteria.

## Methodological safeguards

FY2018 outcomes must not be used for model retraining,
hyperparameter tuning, feature selection, or calibration.

FY2019 remains untouched until the final model and
evaluation rules are frozen.

The dataset is retrospective and status-selected.
Loan term and its derived invalid-term indicator were
excluded because of unresolved approval-time provenance
and potential target leakage.

## Reproducibility note (repository review)

Run after Notebook 04, using Python 3.12.8 and scikit-learn 1.5.1, from the `notebooks/` working directory. The final model-selection record is intentionally created before the FY2019 holdout test. Saved output values were not re-executed during this review. The bootstrap uses loan-level resampling, not lender-clustered resampling.


In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

DATA_PATH = Path("../data/exports/sba_model_export.csv")
MODEL_DIR = Path("../models")

# Fail with actionable guidance if this notebook is started from the wrong folder.
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Required SQL modeling export not found: {DATA_PATH}. "
        "Run this notebook from notebooks/ after generating data/exports/sba_model_export.csv."
    )

_required_models = (
    "logistic_regression_13_features.joblib",
    "gradient_boosting_13_features.joblib",
    "logistic_regression_13_features_calibrated.joblib",
    "gradient_boosting_13_features_calibrated.joblib",
)
for _model_file in _required_models:
    if not (MODEL_DIR / _model_file).is_file():
        raise FileNotFoundError(
            f"Required saved model missing: {MODEL_DIR / _model_file}. "
            "Run the preceding modeling/calibration notebooks first."
        )


In [2]:
# Load the saved calibrated models
lr_calibrated_model = joblib.load(
    MODEL_DIR / "logistic_regression_13_features_calibrated.joblib"
)

gb_calibrated_model = joblib.load(
    MODEL_DIR / "gradient_boosting_13_features_calibrated.joblib"
)

print("Calibrated Logistic Regression loaded.")
print("Calibrated Gradient Boosting loaded.")

# Verify fitted calibration objects
assert len(lr_calibrated_model.calibrated_classifiers_) == 1
assert len(gb_calibrated_model.calibrated_classifiers_) == 1

print("\nFrozen calibrated models validated!")

Calibrated Logistic Regression loaded.
Calibrated Gradient Boosting loaded.

Frozen calibrated models validated!


In [3]:
# Load the full modeling export
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["approval_date"],
    dtype={
        "sba_lender_id": "string",
        "lender_key": "string"
    }
)

# Isolate the independent FY2018 validation cohort
validate_df = df.loc[
    df["modeling_period"] == "Validate"
].copy()

# Display cohort information
print("Full dataset shape:", df.shape)
print("FY2018 validation shape:", validate_df.shape)

print(
    "FY2018 charge-offs:",
    validate_df["chargeoff_within_5yr"].sum()
)

print(
    "FY2018 event incidence:",
    f"{validate_df['chargeoff_within_5yr'].mean():.4%}"
)

# Validate cohort integrity
assert df.shape == (478953, 25)
assert len(validate_df) == 54194
assert validate_df["chargeoff_within_5yr"].sum() == 2529
assert validate_df["approval_fy"].eq(2018).all()

print("\nFY2018 validation cohort checks passed!")

Full dataset shape: (478953, 25)
FY2018 validation shape: (54194, 25)
FY2018 charge-offs: 2529
FY2018 event incidence: 4.6666%

FY2018 validation cohort checks passed!


In [4]:
numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "initial_interest_rate_model",
    "jobs_supported"
]

categorical_features = [
    "business_age_group",
    "naics_sector",
    "project_state",
    "processing_method",
    "collateral_indicator",
    "revolver_status",
    "interest_rate_type",
    "business_type"
]

flag_features = [
    "invalid_interest_rate_flag"
]

feature_columns = (
    numeric_features
    + categorical_features
    + flag_features
)

target_column = "chargeoff_within_5yr"

print("Number of predictors:", len(feature_columns))

assert len(feature_columns) == 13
assert "term_months_model" not in feature_columns
assert "invalid_term_flag" not in feature_columns

print("Feature selection validated!")

Number of predictors: 13
Feature selection validated!


In [5]:
# Select FY2018 predictors and actual outcomes
X_validate_lr = validate_df[feature_columns].copy()

y_validate = validate_df[target_column].copy()

print("Logistic Regression inputs:", X_validate_lr.shape)
print("Actual outcomes:", y_validate.shape)

assert X_validate_lr.shape == (54194, 13)
assert y_validate.shape == (54194,)

print("\nLogistic Regression inputs validated!")

Logistic Regression inputs: (54194, 13)
Actual outcomes: (54194,)

Logistic Regression inputs validated!


In [6]:
# Retrieve FY2010–2015 records for category definitions
train_df = df.loc[
    df["modeling_period"] == "Train"
].copy()

# Prepare independent validation features
X_validate_gb = validate_df[feature_columns].copy()

# Recreate training-derived category mappings
category_mappings = {}

for feature in categorical_features:

    train_categories = pd.Index(
        train_df[feature].dropna().unique()
    )

    category_mappings[feature] = train_categories

    categorical_dtype = pd.CategoricalDtype(
        categories=train_categories
    )

    X_validate_gb[feature] = X_validate_gb[feature].astype(
        categorical_dtype
    )

print("Gradient Boosting categorical dtypes:")

print(X_validate_gb[categorical_features].dtypes)

Gradient Boosting categorical dtypes:
business_age_group      category
naics_sector            category
project_state           category
processing_method       category
collateral_indicator    category
revolver_status         category
interest_rate_type      category
business_type           category
dtype: object


In [7]:
# Confirm categorical data types
assert all(
    isinstance(
        X_validate_gb[feature].dtype,
        pd.CategoricalDtype
    )
    for feature in categorical_features
)

# Confirm feature names and order
assert list(X_validate_gb.columns) == list(
    gb_calibrated_model.calibrated_classifiers_[0]
    .estimator.feature_names_in_
)

# Check unseen categorical values
print("FY2018 unseen category counts:")

for feature in categorical_features:

    unseen_count = (
        validate_df[feature].notna()
        & X_validate_gb[feature].isna()
    ).sum()

    print(f"{feature}: {unseen_count}")

print("\nGradient Boosting feature validation passed!")

FY2018 unseen category counts:
business_age_group: 3948
naics_sector: 0
project_state: 0
processing_method: 2
collateral_indicator: 0
revolver_status: 0
interest_rate_type: 0
business_type: 0

Gradient Boosting feature validation passed!


In [8]:
# Identify business-age categories observed during training
training_age_categories = set(
    train_df["business_age_group"].dropna().unique()
)

# Identify FY2018 values that were absent during training
unseen_age_mask = (
    validate_df["business_age_group"].notna()
    & ~validate_df["business_age_group"].isin(
        training_age_categories
    )
)

# Display unseen categories and their frequencies
unseen_age_summary = (
    validate_df.loc[
        unseen_age_mask,
        "business_age_group"
    ]
    .value_counts(dropna=False)
)

print("Training business-age categories:")
print(sorted(training_age_categories))

print("\nUnseen FY2018 business-age categories:")
print(unseen_age_summary)

print("\nTotal affected records:", unseen_age_mask.sum())

assert unseen_age_mask.sum() == 3948

Training business-age categories:
['2–3 Years', '3–4 Years', '4–5 Years', '5+ Years', 'Over 2 Years (Unspecified)', 'Startup', 'Under 1 Year', 'Unknown']

Unseen FY2018 business-age categories:
business_age_group
Ownership Change    3938
Up to 2 Years         10
Name: count, dtype: int64

Total affected records: 3948


In [9]:
# Load original frozen 13-feature models
lr_original_model = joblib.load(
    MODEL_DIR / "logistic_regression_13_features.joblib"
)

gb_original_model = joblib.load(
    MODEL_DIR / "gradient_boosting_13_features.joblib"
)

# Verify the model configurations
assert lr_original_model.named_steps["classifier"].C == 1.0
assert gb_original_model.learning_rate == 0.05
assert gb_original_model.max_iter == 200
assert gb_original_model.max_leaf_nodes == 31

print("Both original models loaded successfully!")

Both original models loaded successfully!


In [10]:
# Logistic Regression predictions
lr_original_probs = lr_original_model.predict_proba(
    X_validate_lr
)[:, 1]

lr_calibrated_probs = lr_calibrated_model.predict_proba(
    X_validate_lr
)[:, 1]

# Gradient Boosting predictions
gb_original_probs = gb_original_model.predict_proba(
    X_validate_gb
)[:, 1]

gb_calibrated_probs = gb_calibrated_model.predict_proba(
    X_validate_gb
)[:, 1]

# Validate all four probability arrays
prediction_sets = {
    "Logistic Regression - Original": lr_original_probs,
    "Logistic Regression - Calibrated": lr_calibrated_probs,
    "Gradient Boosting - Original": gb_original_probs,
    "Gradient Boosting - Calibrated": gb_calibrated_probs
}

for name, probabilities in prediction_sets.items():

    assert len(probabilities) == len(y_validate)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    print(f"{name}: {len(probabilities)} valid predictions")

print("\nAll FY2018 prediction validations passed!")

Logistic Regression - Original: 54194 valid predictions
Logistic Regression - Calibrated: 54194 valid predictions
Gradient Boosting - Original: 54194 valid predictions
Gradient Boosting - Calibrated: 54194 valid predictions

All FY2018 prediction validations passed!


In [11]:
def evaluate_model(y_true, probabilities):

    y = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    # Calculate the number of loans that can be reviewed
    review_count = int(np.ceil(len(y) * 0.10))

    # Identify the highest-risk loans
    top_indices = np.argsort(
        -probabilities,
        kind="stable"
    )[:review_count]

    # Count actual charge-offs captured
    captured = int(y[top_indices].sum())

    # Calculate evaluation metrics
    return {
        "Average_Precision": average_precision_score(
            y, probabilities
        ),
        "ROC_AUC": roc_auc_score(
            y, probabilities
        ),
        "Recall_at_10": captured / y.sum(),
        "Precision_at_10": captured / review_count,
        "Captured_Chargeoffs": captured,
        "Brier_Score": brier_score_loss(
            y, probabilities
        ),
        "Average_Predicted_Risk": probabilities.mean()
    }

In [12]:
validation_results = []

for model_name, probabilities in prediction_sets.items():

    metrics = evaluate_model(
        y_validate,
        probabilities
    )

    validation_results.append({
        "Model": model_name,
        **metrics
    })

validation_df = pd.DataFrame(validation_results)

print("FY2018 Independent Validation Results:")

display(validation_df.round(4))

FY2018 Independent Validation Results:


,Model,Average_Precision,ROC_AUC,Recall_at_10,Precision_at_10,Captured_Chargeoffs,Brier_Score,Average_Predicted_Risk
0,Logistic Regression - Original,0.0941,0.7076,0.2262,0.1055,572,0.0435,0.0515
1,Logistic Regression - Calibrated,0.0941,0.7076,0.2262,0.1055,572,0.0437,0.0596
2,Gradient Boosting - Original,0.0924,0.7030,0.2313,0.1079,585,0.0435,0.0422
3,Gradient Boosting - Calibrated,0.0924,0.7030,0.2313,0.1079,585,0.0435,0.0493


In [13]:
# Validate the evaluation table
assert len(validation_df) == 4

assert validation_df["Average_Precision"].between(0, 1).all()
assert validation_df["ROC_AUC"].between(0, 1).all()
assert validation_df["Recall_at_10"].between(0, 1).all()
assert validation_df["Precision_at_10"].between(0, 1).all()
assert validation_df["Brier_Score"].between(0, 1).all()

print("FY2018 loans:", len(y_validate))
print("FY2018 charge-offs:", int(y_validate.sum()))
print("Review capacity:", int(np.ceil(len(y_validate) * 0.10)))
print(f"Observed incidence: {y_validate.mean():.4%}")

print("\nIndependent validation calculations passed!")

FY2018 loans: 54194
FY2018 charge-offs: 2529
Review capacity: 5420
Observed incidence: 4.6666%

Independent validation calculations passed!


In [14]:
# Reference probability is based on FY2010–2015 training only
train_prevalence = df.loc[
    df["modeling_period"] == "Train",
    "chargeoff_within_5yr"
].mean()

reference_probs = np.full(
    len(y_validate),
    train_prevalence
)

reference_brier = brier_score_loss(
    y_validate,
    reference_probs
)

print(f"Training prevalence: {train_prevalence:.4%}")
print(f"Reference Brier: {reference_brier:.8f}")

print("\nFY2018 probability evaluation:")

for name, probabilities in prediction_sets.items():

    brier = brier_score_loss(
        y_validate,
        probabilities
    )

    skill = 1 - brier / reference_brier

    print(
        f"{name}: "
        f"Brier={brier:.8f}, "
        f"Skill={skill:.4%}, "
        f"Average risk={probabilities.mean():.4%}"
    )

Training prevalence: 3.7164%
Reference Brier: 0.04457828

FY2018 probability evaluation:
Logistic Regression - Original: Brier=0.04351223, Skill=2.3914%, Average risk=5.1525%
Logistic Regression - Calibrated: Brier=0.04374384, Skill=1.8719%, Average risk=5.9647%
Gradient Boosting - Original: Brier=0.04352805, Skill=2.3559%, Average risk=4.2191%
Gradient Boosting - Calibrated: Brier=0.04350737, Skill=2.4023%, Average risk=4.9331%


In [15]:
def calculate_calibration_deciles(y_true, probabilities):

    calibration_df = pd.DataFrame({
        "actual": np.asarray(y_true),
        "predicted": np.asarray(probabilities)
    })

    # Create approximately equal-sized predicted-risk groups
    calibration_df["decile"] = (
        pd.qcut(
            calibration_df["predicted"].rank(method="first"),
            q=10,
            labels=False
        ) + 1
    )

    summary = (
        calibration_df
        .groupby("decile")
        .agg(
            loan_count=("actual", "size"),
            chargeoffs=("actual", "sum"),
            avg_predicted_risk=("predicted", "mean"),
            observed_chargeoff_rate=("actual", "mean")
        )
        .reset_index()
    )

    summary["calibration_gap"] = (
        summary["observed_chargeoff_rate"]
        - summary["avg_predicted_risk"]
    )

    return summary

In [16]:
validation_deciles = {}

for model_name, probabilities in prediction_sets.items():

    summary = calculate_calibration_deciles(
        y_validate,
        probabilities
    )

    validation_deciles[model_name] = summary

    # Validate group totals
    assert len(summary) == 10
    assert summary["loan_count"].sum() == 54194
    assert summary["chargeoffs"].sum() == 2529

print("All FY2018 calibration deciles validated!")

All FY2018 calibration deciles validated!


In [17]:
percentage_columns = [
    "avg_predicted_risk",
    "observed_chargeoff_rate",
    "calibration_gap"
]

for version in [
    "Logistic Regression - Original",
    "Logistic Regression - Calibrated"
]:

    display_df = validation_deciles[version].copy()

    display_df[percentage_columns] *= 100

    print(f"\n{version}")
    display(display_df.round(3))


Logistic Regression - Original


,decile,loan_count,chargeoffs,avg_predicted_risk,observed_chargeoff_rate,calibration_gap
0,1,5420,23,0.745,0.424,-0.321
1,2,5419,53,1.825,0.978,-0.847
2,3,5419,93,2.604,1.716,-0.888
3,4,5420,150,3.305,2.768,-0.538
4,5,5419,214,4.019,3.949,-0.069
5,6,5419,245,4.801,4.521,-0.280
6,7,5420,312,5.690,5.756,0.066
7,8,5419,398,6.795,7.345,0.550
8,9,5419,469,8.549,8.655,0.106
9,10,5420,572,13.193,10.554,-2.639



Logistic Regression - Calibrated


,decile,loan_count,chargeoffs,avg_predicted_risk,observed_chargeoff_rate,calibration_gap
0,1,5420,23,0.832,0.424,-0.407
1,2,5419,53,2.070,0.978,-1.092
2,3,5419,93,2.975,1.716,-1.259
3,4,5420,150,3.794,2.768,-1.027
4,5,5419,214,4.629,3.949,-0.680
5,6,5419,245,5.547,4.521,-1.026
6,7,5420,312,6.591,5.756,-0.835
7,8,5419,398,7.890,7.345,-0.545
8,9,5419,469,9.949,8.655,-1.295
9,10,5420,572,15.369,10.554,-4.815


In [18]:
for version in [
    "Gradient Boosting - Original",
    "Gradient Boosting - Calibrated"
]:

    display_df = validation_deciles[version].copy()

    display_df[percentage_columns] *= 100

    print(f"\n{version}")
    display(display_df.round(3))


Gradient Boosting - Original


,decile,loan_count,chargeoffs,avg_predicted_risk,observed_chargeoff_rate,calibration_gap
0,1,5420,23,0.733,0.424,-0.309
1,2,5419,53,1.462,0.978,-0.484
2,3,5419,111,2.147,2.048,-0.098
3,4,5420,159,2.780,2.934,0.154
4,5,5419,205,3.409,3.783,0.374
5,6,5419,249,4.066,4.595,0.529
6,7,5420,329,4.799,6.070,1.271
7,8,5419,371,5.696,6.846,1.150
8,9,5419,444,6.993,8.193,1.201
9,10,5420,585,10.106,10.793,0.687



Gradient Boosting - Calibrated


,decile,loan_count,chargeoffs,avg_predicted_risk,observed_chargeoff_rate,calibration_gap
0,1,5420,23,0.795,0.424,-0.371
1,2,5419,53,1.629,0.978,-0.651
2,3,5419,111,2.431,2.048,-0.383
3,4,5420,159,3.181,2.934,-0.248
4,5,5419,205,3.933,3.783,-0.150
5,6,5419,249,4.722,4.595,-0.127
6,7,5420,329,5.607,6.070,0.463
7,8,5419,371,6.694,6.846,0.152
8,9,5419,444,8.272,8.193,-0.079
9,10,5420,585,12.066,10.793,-1.272


In [19]:
# Freeze FY2018 outcomes and the two calibrated prediction arrays
y_boot = y_validate.to_numpy()

lr_boot_probs = lr_calibrated_probs.copy()
gb_boot_probs = gb_calibrated_probs.copy()

# Bootstrap settings
N_BOOTSTRAPS = 500
rng = np.random.default_rng(RANDOM_STATE)

bootstrap_results = []

for iteration in range(N_BOOTSTRAPS):

    # Sample loan positions with replacement
    sample_idx = rng.choice(
        len(y_boot),
        size=len(y_boot),
        replace=True
    )

    # Apply identical positions to both models
    y_sample = y_boot[sample_idx]

    lr_sample = lr_boot_probs[sample_idx]
    gb_sample = gb_boot_probs[sample_idx]

    # Skip samples without both outcome classes
    if len(np.unique(y_sample)) < 2:
        continue

    # Calculate Average Precision
    lr_ap = average_precision_score(y_sample, lr_sample)
    gb_ap = average_precision_score(y_sample, gb_sample)

    # Calculate Recall@10% separately for each model
    review_n = int(np.ceil(len(y_sample) * 0.10))

    lr_top = np.argsort(
        -lr_sample,
        kind="stable"
    )[:review_n]

    gb_top = np.argsort(
        -gb_sample,
        kind="stable"
    )[:review_n]

    lr_recall = y_sample[lr_top].sum() / y_sample.sum()
    gb_recall = y_sample[gb_top].sum() / y_sample.sum()

    bootstrap_results.append({
        "AP_Difference": gb_ap - lr_ap,
        "Recall_Difference": gb_recall - lr_recall
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

print("Bootstrap samples completed:", len(bootstrap_df))

Bootstrap samples completed: 500


In [20]:
# Calculate percentile-based 95% bootstrap intervals
bootstrap_summary = pd.DataFrame({
    "Metric": [
        "Average Precision Difference",
        "Recall@10% Difference"
    ],
    "Mean_Difference": [
        bootstrap_df["AP_Difference"].mean(),
        bootstrap_df["Recall_Difference"].mean()
    ],
    "CI_Lower": [
        bootstrap_df["AP_Difference"].quantile(0.025),
        bootstrap_df["Recall_Difference"].quantile(0.025)
    ],
    "CI_Upper": [
        bootstrap_df["AP_Difference"].quantile(0.975),
        bootstrap_df["Recall_Difference"].quantile(0.975)
    ]
})

display(bootstrap_summary.round(5))

assert len(bootstrap_df) == N_BOOTSTRAPS

print("\nBootstrap uncertainty analysis validated!")

,Metric,Mean_Difference,CI_Lower,CI_Upper
0,Average Precision Difference,-0.00205,-0.00761,0.00348
1,Recall@10% Difference,0.00454,-0.01302,0.02249



Bootstrap uncertainty analysis validated!


In [21]:
stability_df = pd.DataFrame({
    "Period": ["FY2016 - Tune", "FY2017 - Calibrate", "FY2018 - Validate"],
    "LR_AP": [0.0805, 0.0861, 0.0941],
    "GB_AP": [0.0923, 0.1002, 0.0924],
    "LR_Recall_10": [0.2370, 0.2220, 0.2262],
    "GB_Recall_10": [0.2734, 0.2666, 0.2313]
})

stability_df["AP_Difference"] = (
    stability_df["GB_AP"] - stability_df["LR_AP"]
)

stability_df["Recall_Difference"] = (
    stability_df["GB_Recall_10"] - stability_df["LR_Recall_10"]
)

print("Chronological Model Stability Summary:")
display(stability_df.round(4))

Chronological Model Stability Summary:


,Period,LR_AP,GB_AP,LR_Recall_10,GB_Recall_10,AP_Difference,Recall_Difference
0,FY2016 - Tune,0.0805,0.0923,0.2370,0.2734,0.0118,0.0364
1,FY2017 - Calibrate,0.0861,0.1002,0.2220,0.2666,0.0141,0.0446
2,FY2018 - Validate,0.0941,0.0924,0.2262,0.2313,-0.0017,0.0051


In [22]:
# Record the final model-selection decision before FY2019 testing
model_selection_record = {
    "selected_model": "Logistic Regression",
    "selected_artifact": "logistic_regression_13_features.joblib",
    "predictor_count": 13,
    "regularization_C": 1.0,
    "probability_calibration": "Uncalibrated",
    "training_period": "FY2010-2015",
    "tuning_period": "FY2016",
    "calibration_period": "FY2017",
    "validation_period": "FY2018",
    "test_period": "FY2019",
    "review_capacity": 0.10,
    "challenger_selected": False,
    "selection_reason": (
        "Gradient Boosting failed the predefined FY2018 "
        "Average Precision and Recall@10% improvement thresholds."
    ),
    "calibration_note": (
        "FY2017 sigmoid calibration worsened Logistic Regression "
        "Brier score on FY2018. Uncalibrated probabilities retained; "
        "this choice was informed by FY2018 validation."
    ),
    "feature_provenance_status": "Provisional",
    "test_evaluated": False
}

selection_df = pd.DataFrame(
    model_selection_record.items(),
    columns=["Decision_Item", "Value"]
)

display(selection_df)

,Decision_Item,Value
0,selected_model,Logistic Regression
1,selected_artifact,logistic_regression_13_features.joblib
2,predictor_count,13
3,regularization_C,1.0
4,probability_calibration,Uncalibrated
5,training_period,FY2010-2015
6,tuning_period,FY2016
7,calibration_period,FY2017
8,validation_period,FY2018
9,test_period,FY2019


In [23]:
import json
from datetime import datetime, timezone

selection_path = MODEL_DIR / "model_selection_record.json"

model_selection_record["record_created_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

with open(selection_path, "w") as file:
    json.dump(
        model_selection_record,
        file,
        indent=4
    )

print("Selection record saved:", selection_path.exists())
print("Selected model:", model_selection_record["selected_model"])
print("Test evaluated:", model_selection_record["test_evaluated"])

assert selection_path.exists()
assert model_selection_record["test_evaluated"] is False

print("\nModel-selection decision recorded successfully!")

Selection record saved: True
Selected model: Logistic Regression
Test evaluated: False

Model-selection decision recorded successfully!
